In [ ]:
import torch
import numpy as np


def ellipse_perimeter(a, b, pi, one, three, ten, four, c1=None, c2=None):
    h = ((a - b) ** 2) / ((a + b) ** 2)
    base = (
        pi
        * (a + b)
        * (one + (three * h) / (ten + torch.sqrt(torch.relu(four - three * h))))
    )
    if c1 is not None and c2 is not None:
        base = base + c1 * h**2 + c2 * h**3
    return base


def _gl_nodes_weights(n: int, dtype, device):
    nodes, weights = np.polynomial.legendre.leggauss(n)
    return (
        torch.tensor(nodes, dtype=dtype, device=device),
        torch.tensor(weights, dtype=dtype, device=device),
    )


def recursive_perimeter(a, b, n: int = 64):
    nodes, weights = _gl_nodes_weights(n, dtype=a.dtype, device=a.device)

    theta = (torch.pi / 4) * (nodes + 1)
    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)

    if a.dim() > 0:
        a = a.unsqueeze(-1)
        b = b.unsqueeze(-1)

    integrand = torch.sqrt(a**2 * cos_t**2 + b**2 * sin_t**2)
    return 4 * (torch.pi / 4) * (weights * integrand).sum(-1)

In [ ]:
def compute_loss(approximator, a, b):
    constants = tuple(approximator[0])
    true_p = recursive_perimeter(a, b)
    approx_p = ellipse_perimeter(a, b, *constants)
    return ((true_p - approx_p) ** 2).mean()


ellipse_approximator = torch.Tensor([3, 1, 3, 10, 4, 0, 0]).unsqueeze(0)

In [ ]:
sample_size = 1000
epochs = 10

ellipses = torch.stack(
    [
        torch.Tensor([a, b])
        for (a, b) in list(
            set(
                [
                    (x + 1, y * y // (x + 1) + 1)
                    for (x, y) in zip(range(sample_size), range(sample_size))
                ]
                + [(1, x + 1) for x in range(sample_size)]
            )
        )
    ]
    * epochs
)

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader


def train(ea, s, batch_size=256):
    ea = ea.detach().clone().float().requires_grad_(True)

    optimizer = torch.optim.Adam([ea], lr=1e-4)
    loader = DataLoader(s, batch_size=batch_size, shuffle=True)
    pbar = tqdm(loader, desc="Training")

    for t in pbar:
        optimizer.zero_grad()
        a, b = t[:, 0].to(ea.device), t[:, 1].to(ea.device)
        loss = compute_loss(ea, a, b)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(mse=loss.item())

    return ea


ellipse_approximator = train(ellipse_approximator.to("cuda"), ellipses)

In [ ]:
test_ellipse = torch.Tensor([1, 15]).to("cuda")

with torch.no_grad():
    ramanujan = (
        5
        - compute_loss(
            torch.tensor([torch.pi, 1.0, 3.0, 10.0, 4.0], dtype=torch.float32)
            .to("cuda")
            .unsqueeze(0),
            test_ellipse[0],
            test_ellipse[1],
        )
    ).item()
    me = (
        5
        - compute_loss(ellipse_approximator.detach(), test_ellipse[0], test_ellipse[1])
    ).item()

while ramanujan >= me:
    ellipse_approximator = train(ellipse_approximator, ellipses.to("cuda"))
    with torch.no_grad():
        me = (
            5
            - compute_loss(
                ellipse_approximator.detach(), test_ellipse[0], test_ellipse[1]
            )
        ).item()

print(ramanujan, "to", me)